# GRPO Fine-Tuning for CSE 151B Math Reasoning

This notebook fine-tunes `Qwen/Qwen3-4B-Thinking-2507` with TRL's `GRPOTrainer`. It mirrors the starter evaluation path: MCQ rewards use boxed-letter exact match, and free-form rewards use the local `Judger.auto_judge()` symbolic/numeric evaluator.

## 1. Environment

Install these packages in the same environment/kernel used for training. The first cell is intentionally commented so rerunning the notebook does not reinstall packages by accident.

In [ ]:
# Uncomment for a fresh environment, then restart the kernel.
# !python -m pip install -U \
#     "trl>=0.17.0" \
#     "transformers>=4.51.0" \
#     accelerate datasets peft bitsandbytes safetensors \
#     sympy numpy tqdm antlr4-python3-runtime==4.11.1 ipykernel jupyter

## 2. Imports and Configuration

In [ ]:
import json
import os
import random
import re
import sys
from pathlib import Path
from typing import Any, Optional

import torch
from datasets import Dataset
from peft import LoraConfig
from tqdm.auto import tqdm
from transformers import AutoTokenizer, BitsAndBytesConfig
from trl import GRPOConfig, GRPOTrainer

sys.path.insert(0, ".")
from judger import Judger

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
DATA_PATH = Path("data/public.jsonl")
OUTPUT_DIR = Path("outputs/grpo-qwen3-4b-thinking")
RESULTS_PATH = Path("results/grpo_eval_samples.jsonl")

SEED = 151
GPU_ID = "0"
USE_QLORA = True
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16

# Keep the default path cheap and safe. Set these to False/True for a real run.
DEBUG_SUBSET = True
DEBUG_TRAIN_SIZE = 32
DEBUG_EVAL_SIZE = 16
RUN_TRAINING = False
RUN_POST_TRAIN_EVAL = False

TRAIN_TEST_SPLIT = 0.08
MAX_PROMPT_LENGTH = 2048
MAX_COMPLETION_LENGTH = 1024
NUM_GENERATIONS = 4

random.seed(SEED)
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"bf16: {USE_BF16}, fp16: {USE_FP16}")

## 3. Prompt Construction

The prompts intentionally match the starter notebook's format so the trained policy is rewarded for behavior that the competition scorer can extract.

In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list[str]]) -> list[dict[str, str]]:
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{label}. {option.strip()}" for label, option in zip(labels, options))
        user_prompt = f"{question}\n\nOptions:\n{opts_text}"
        return [
            {"role": "system", "content": SYSTEM_PROMPT_MCQ},
            {"role": "user", "content": user_prompt},
        ]

    return [
        {"role": "system", "content": SYSTEM_PROMPT_MATH},
        {"role": "user", "content": question},
    ]

## 4. Dataset Loading and Split

In [ ]:
def load_public_dataset(path: Path) -> Dataset:
    rows = []
    with path.open() as f:
        for line in f:
            item = json.loads(line)
            options = item.get("options") or []
            rows.append(
                {
                    "id": item["id"],
                    "prompt": build_prompt(item["question"], options),
                    "question": item["question"],
                    "options": options,
                    "answer": item["answer"],
                    "is_mcq": bool(options),
                }
            )
    return Dataset.from_list(rows)


full_dataset = load_public_dataset(DATA_PATH).shuffle(seed=SEED)
split = full_dataset.train_test_split(test_size=TRAIN_TEST_SPLIT, seed=SEED)
train_dataset = split["train"]
eval_dataset = split["test"]

if DEBUG_SUBSET:
    train_dataset = train_dataset.select(range(min(DEBUG_TRAIN_SIZE, len(train_dataset))))
    eval_dataset = eval_dataset.select(range(min(DEBUG_EVAL_SIZE, len(eval_dataset))))

print(train_dataset)
print(eval_dataset)
print(train_dataset[0]["prompt"][-1]["content"][:500])

## 5. Reward Functions

`GRPOTrainer` calls custom rewards with `prompts`, `completions`, and every non-`prompt` dataset column. These functions accept `**kwargs` for compatibility with TRL's reward API.

In [ ]:
judger = Judger(strict_extract=False)


def completion_to_text(completion: Any) -> str:
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list) and completion and isinstance(completion[0], dict):
        return completion[0].get("content", "")
    return str(completion)


def extract_boxed_text(text: str) -> str:
    try:
        return judger.extract_boxed_answer(text).strip()
    except Exception:
        return ""


def extract_letter(text: str) -> str:
    boxed = extract_boxed_text(text)
    match = re.search(r"\b([A-Za-z])\b", boxed)
    if match:
        return match.group(1).upper()

    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def correctness_reward(completions, answer, is_mcq, options=None, **kwargs):
    rewards = []
    for completion, gold, mcq in zip(completions, answer, is_mcq):
        text = completion_to_text(completion)
        if mcq:
            rewards.append(1.0 if extract_letter(text) == str(gold).strip().upper() else 0.0)
            continue

        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=text,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False
        rewards.append(1.0 if correct else 0.0)
    return rewards


def boxed_format_reward(completions, **kwargs):
    rewards = []
    for completion in completions:
        text = completion_to_text(completion)
        boxed = extract_boxed_text(text)
        rewards.append(1.0 if boxed else 0.0)
    return rewards


def reasoning_structure_reward(completions, **kwargs):
    rewards = []
    for completion in completions:
        text = completion_to_text(completion)
        has_final = bool(re.search(r"final answer|therefore|\\boxed", text, flags=re.IGNORECASE))
        has_work = len(text.split()) >= 20
        rewards.append(1.0 if has_final and has_work else 0.0)
    return rewards


# Quick reward sanity checks against the same evaluator logic used by the starter notebook.
sample_rewards = correctness_reward(
    completions=[r"The final answer is \\boxed{105950}.", r"I choose \\boxed{F}.", r"\\boxed{A}"],
    answer=[["325*(1+325)"], "F", "C"],
    is_mcq=[False, True, True],
)
print(sample_rewards)

## 6. Tokenizer and Training Configuration

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_init_kwargs = {
    "trust_remote_code": True,
    "torch_dtype": torch.bfloat16 if USE_BF16 else torch.float16,
    "attn_implementation": "sdpa",
}

if USE_QLORA:
    model_init_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    )

peft_config = None
if USE_QLORA:
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )

training_args = GRPOConfig(
    output_dir=str(OUTPUT_DIR),
    run_name="qwen3-4b-thinking-grpo-cse151b",
    seed=SEED,
    data_seed=SEED,
    bf16=USE_BF16,
    fp16=USE_FP16,
    gradient_checkpointing=True,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    max_steps=20 if DEBUG_SUBSET else 300,
    warmup_ratio=0.03,
    logging_steps=1 if DEBUG_SUBSET else 10,
    save_steps=10 if DEBUG_SUBSET else 50,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=10 if DEBUG_SUBSET else 50,
    report_to="none",
    remove_unused_columns=False,
    max_prompt_length=MAX_PROMPT_LENGTH,
    max_completion_length=MAX_COMPLETION_LENGTH,
    num_generations=NUM_GENERATIONS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    beta=0.04,
    reward_weights=[1.0, 0.2, 0.1],
    log_completions=True,
    model_init_kwargs=model_init_kwargs,
)

training_args

## 7. Trainer Initialization

This cell loads the model. In debug mode, it is still the expensive step; skip it if you only want to inspect data and reward functions.

In [ ]:
trainer = GRPOTrainer(
    model=MODEL_ID,
    args=training_args,
    reward_funcs=[correctness_reward, boxed_format_reward, reasoning_structure_reward],
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

print("Trainer initialized.")

## 8. Training

Set `RUN_TRAINING = True` in the configuration cell when ready. The default deliberately avoids launching a full training job.

In [ ]:
if RUN_TRAINING:
    train_result = trainer.train()
    trainer.save_model(str(OUTPUT_DIR / "final"))
    tokenizer.save_pretrained(str(OUTPUT_DIR / "final"))
    train_result
else:
    print("RUN_TRAINING is False; skipping training.")

## 9. Lightweight Post-Training Evaluation

This samples a few eval prompts, generates responses with the trainer model, and scores them through the same reward/evaluator path. It is gated separately from training.

In [ ]:
def score_response(response: str, item: dict[str, Any]) -> bool:
    if item["is_mcq"]:
        return extract_letter(response) == str(item["answer"]).strip().upper()

    gold_list = item["answer"] if isinstance(item["answer"], list) else [item["answer"]]
    try:
        return judger.auto_judge(pred=response, gold=gold_list, options=[[]] * len(gold_list))
    except Exception:
        return False


if RUN_POST_TRAIN_EVAL:
    model = trainer.model
    model.eval()
    records = []
    for item in tqdm(eval_dataset.select(range(min(8, len(eval_dataset))))):
        prompt_text = tokenizer.apply_chat_template(
            item["prompt"],
            tokenize=False,
            add_generation_prompt=True,
        )
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_COMPLETION_LENGTH,
                do_sample=True,
                temperature=0.6,
                top_p=0.95,
            )
        response = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
        records.append(
            {
                "id": item["id"],
                "is_mcq": item["is_mcq"],
                "gold": item["answer"],
                "response": response,
                "correct": score_response(response, item),
            }
        )

    with RESULTS_PATH.open("w") as f:
        for record in records:
            f.write(json.dumps(record) + "\n")

    print(f"Saved {len(records)} sampled eval records to {RESULTS_PATH}")
    print(f"Sample accuracy: {sum(r['correct'] for r in records)} / {len(records)}")
else:
    print("RUN_POST_TRAIN_EVAL is False; skipping generation eval.")

## Notes and Assumptions

- The training dataset is currently `data/public.jsonl`, split into train/eval subsets.
- QLoRA is enabled by default because full fine-tuning a 4B reasoning model is likely memory-heavy.
- `RUN_TRAINING` and `RUN_POST_TRAIN_EVAL` default to `False` so opening or running the notebook top-to-bottom will not accidentally launch a long job.
- Reward correctness intentionally reuses `Judger.auto_judge()` for free-form answers to match the starter evaluator as closely as possible.